In [6]:
# ============================================================
# IG_Exploration.ipynb — Complete Integrated Gradients Notebook
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import sys
ROOT = Path().resolve().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from libs.utils.importance import (
    integrated_gradients_compare,
    ordinal_logits_to_probs
)

from libs.diagnostics.batch_ig import run_batch_ig
from libs.diagnostics.block_summary import blockwise_ig, compare_blocks
from libs.diagnostics.error_clusters import cluster_misclassifications, summarize_cluster
from libs.diagnostics.ig_html_dashboard import make_ig_bar_chart, make_block_level_chart

from libs.models.io_config import load_yaml, load_prep_artifacts, read_arrow_matrix
from libs.utils.importance import get_block_schema

import matplotlib.pyplot as plt

# ---------------------------------------
# CONFIG
# ---------------------------------------
PREP_ID = "pca225_tv0.95_sc103261f6_nt95_fmB1"
RUN = "clf_mlp_B1_20251208_174307"
MODEL_PATH = f"artifacts/models/severity/{PREP_ID}/{RUN}/best_model.pt"

X_PATH = f"artifacts/prep/{PREP_ID}/X_val.arrow"
Y_PATH = f"artifacts/prep/{PREP_ID}/y_val.npy"
SCHEMA_PATH = f"artifacts/prep/{PREP_ID}/schema.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------
# LOAD MODEL & DATA
# ---------------------------------------
model = torch.load(MODEL_PATH, map_location=DEVICE).to(DEVICE)
model.eval()

X_val = torch.tensor(read_arrow_matrix(X_PATH), dtype=torch.float32)
y_arr = np.load(Y_PATH, allow_pickle=True)

classes = sorted(pd.unique(pd.Series(y_arr)))
cls_to_idx = {c:i for i,c in enumerate(classes)}
y_val = torch.tensor([cls_to_idx[c] for c in y_arr], dtype=torch.long)

schema = json.load(open(SCHEMA_PATH))
block_schema = get_block_schema(schema)

feature_names = []
for blk in block_schema:
    for j in range(blk["start"], blk["end"]):
        feature_names.append(f"{blk['name']}_{j - blk['start']}")


# ---------------------------------------
# Pick one sample for IG
# ---------------------------------------
i = np.random.randint(0, len(X_val))
xs = X_val[i]
yt = int(y_val[i])

pred, true, ig_pred, ig_true = integrated_gradients_compare(
    model, xs, true_class=yt, device=DEVICE
)

print("Predicted:", pred, "True:", true)


# ---------------------------------------
# Feature-level plot
# ---------------------------------------
plt.figure(figsize=(16,4))
plt.plot(ig_pred, label="Pred IG")
plt.plot(ig_true, label="True IG")
plt.legend()
plt.title("IG Feature Comparison")
plt.show()


# ---------------------------------------
# Block-level contribution
# ---------------------------------------
bp, bt, diff = compare_blocks(ig_pred, ig_true, block_schema)
print("Block IG (pred):", bp)
print("Block IG (true):", bt)
print("Block difference:", diff)


# ---------------------------------------
# Batch IG diagnostics
# ---------------------------------------
records = run_batch_ig(
    model,
    X_val,
    y_val,
    limit=200,
    only_errors=True,
    device=DEVICE
)

print("Collected misclassified samples:", len(records))


# ---------------------------------------
# Error clusters
# ---------------------------------------
clusters = cluster_misclassifications(records)
print("Error clusters detected:")
for key in clusters:
    print(f" - {key}: {len(clusters[key])} samples")

# summarize each cluster
cluster_summaries = {}
for key, cl in clusters.items():
    cluster_summaries[key] = summarize_cluster(cl, block_schema)

cluster_summaries


ModuleNotFoundError: No module named 'libs'